In [3]:
# =============================================================================
# 1. Import Libraries
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression

from imblearn.over_sampling import SMOTE

import xgboost as xgb
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from tabpfn import TabPFNClassifier
from tabpfn_extensions.post_hoc_ensembles.sklearn_interface import AutoTabPFNClassifier

In [5]:
# =============================================================================
# 2. Load Data
# =============================================================================
train = pd.read_csv('/content/train_processed_v4.csv')
test = pd.read_csv('/content/test_processed_v4.csv')

# =============================================================================
# 3. Data Preprocessing
# =============================================================================
X = train.drop(['No-show', 'PatientId', 'AppointmentID'], axis=1)
y = train['No-show']

categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

# Create a preprocessing pipeline for numerical and categorical features.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numerical_cols),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols)
    ]
)


X_processed = preprocessor.fit_transform(X)

if 'No-show' in test.columns:
    X_test_processed = preprocessor.transform(test.drop('No-show', axis=1))
else:
    X_test_processed = preprocessor.transform(test)

Categorical columns: ['ScheduledDay_Weekday', 'AppointmentDay_Weekday']
Numerical columns: ['Age', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'WaitTime_Log', 'Gender_Encoded', 'ScheduledDay_Month', 'AppointmentDay_Month', 'KModes_Cluster', 'HDBSCAN_Cluster']


In [6]:
# =============================================================================
# 4. Propensity Modeling Analysis (Simple Logistic Regression)
# =============================================================================
propensity_model = LogisticRegression(max_iter=1000, random_state=42)
propensity_model.fit(X_processed, y)
propensity_scores = propensity_model.predict_proba(X_processed)[:, 1]

ps_df = pd.DataFrame({
    'Propensity_Score': propensity_scores,
    'No_show': y
})

# Plot the distribution of propensity scores for each group.
plt.figure(figsize=(10, 6))
sns.histplot(data=ps_df, x='Propensity_Score', hue='No_show', bins=30, kde=True)
plt.title('Distribution of Propensity Scores by No-show')
plt.xlabel('Propensity Score')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('propensity_score_distribution.png')
plt.close()

# Print average propensity scores by group.
mean_ps = ps_df.groupby('No_show')['Propensity_Score'].mean()
print("Mean Propensity Scores by Group:")
print(mean_ps)

Mean Propensity Scores by Group:
No_show
0    0.186659
1    0.263039
Name: Propensity_Score, dtype: float64


In [7]:
# =============================================================================
# 5. Resample Data Using SMOTE (for Classification Modeling)
# =============================================================================
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_processed, y)

print(f"Original training set shape: {X_processed.shape}")
print(f"SMOTE-augmented training set shape: {X_resampled.shape}")
print("Class distribution after SMOTE:")
print(pd.Series(y_resampled).value_counts(normalize=True))

X_train_cls = X_resampled
y_train_cls = y_resampled

Original training set shape: (88421, 25)
SMOTE-augmented training set shape: (141106, 25)
Class distribution after SMOTE:
No-show
0    0.5
1    0.5
Name: proportion, dtype: float64


In [8]:
# =============================================================================
# 6. XGBoost Model with Bayesian Optimization
# =============================================================================
xgb_model = xgb.XGBClassifier(objective='binary:logistic',
                              use_label_encoder=False,
                              eval_metric='logloss',
                              random_state=42)

param_space = {
    'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'n_estimators': Integer(50, 500),
    'max_depth': Integer(3, 10),
    'min_child_weight': Integer(1, 10),
    'gamma': Real(0, 0.5),
    'subsample': Real(0.6, 1.0),
    'colsample_bytree': Real(0.6, 1.0)
}

# Set up Bayesian search with 5-fold cross-validation optimizing F1 score.
xgb_bayes = BayesSearchCV(
    xgb_model,
    param_space,
    n_iter=20,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Fit XGBoost with Bayesian optimization on the resampled training data.
xgb_bayes.fit(X_train_cls, y_train_cls)
print("Best XGBoost parameters found:")
print(xgb_bayes.best_params_)

# Get the best XGBoost model.
best_xgb_model = xgb_bayes.best_estimator_

# -----------------------------------------------------------------------------
# Plot the Bayesian optimization validation curve for XGBoost.
# -----------------------------------------------------------------------------
xgb_results = xgb_bayes.cv_results_
plt.figure(figsize=(10, 6))
plt.plot(xgb_results['mean_test_score'], marker='o')
plt.fill_between(np.arange(len(xgb_results['mean_test_score'])),
                 xgb_results['mean_test_score'] - xgb_results['std_test_score'],
                 xgb_results['mean_test_score'] + xgb_results['std_test_score'],
                 alpha=0.3)
plt.xlabel('Iteration')
plt.ylabel('Mean F1 Score')
plt.title('XGBoost Validation Curve (Bayesian Optimization)')
plt.grid(True)
plt.tight_layout()
plt.savefig('xgboost_validation_curve.png')
plt.close()

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fi

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [22:12:25] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Best XGBoost parameters found:
OrderedDict([('colsample_bytree', 0.8555743293139753), ('gamma', 0.1413882916053488), ('learning_rate', 0.29999999999999993), ('max_depth', 10), ('min_child_weight', 2), ('n_estimators', 500), ('subsample', 0.6854165372861681)])


In [9]:
# =============================================================================
# 7. XGBoost Feature Importance Analysis
# =============================================================================
if hasattr(best_xgb_model, 'feature_importances_'):
    onehot_categories = preprocessor.named_transformers_['cat'].named_steps['onehot'].categories_
    cat_feature_names = [f"{col}_{val}" for col, cats in zip(categorical_cols, onehot_categories) for val in cats]
    feature_names = numerical_cols + cat_feature_names

    feature_importances = best_xgb_model.feature_importances_[:len(feature_names)]
    importance_df = pd.DataFrame({
        'Feature': feature_names[:len(feature_importances)],
        'Importance': feature_importances
    }).sort_values('Importance', ascending=False)

    plt.figure(figsize=(10, 8))
    sns.barplot(x='Importance', y='Feature', data=importance_df.head(20))
    plt.title('Top 20 Feature Importances (XGBoost)')
    plt.tight_layout()
    plt.savefig('feature_importance.png')
    plt.close()

    print("\nTop 10 most important features:")
    print(importance_df.head(10))


Top 10 most important features:
                             Feature  Importance
15     ScheduledDay_Weekday_Saturday    0.126355
7                       WaitTime_Log    0.054576
0                                Age    0.044609
19     AppointmentDay_Weekday_Friday    0.044375
16     ScheduledDay_Weekday_Thursday    0.044286
20     AppointmentDay_Weekday_Monday    0.043270
14       ScheduledDay_Weekday_Monday    0.042662
24  AppointmentDay_Weekday_Wednesday    0.042305
22   AppointmentDay_Weekday_Thursday    0.041112
18    ScheduledDay_Weekday_Wednesday    0.040371


In [10]:
# =============================================================================
# 8. XGBoost Test Set Evaluation
# =============================================================================
if 'No-show' in test.columns:
    y_test = test['No-show']
    y_test_pred_xgb = best_xgb_model.predict(X_test_processed)
    xgb_test_accuracy = accuracy_score(y_test, y_test_pred_xgb)
    xgb_test_precision = precision_score(y_test, y_test_pred_xgb)
    xgb_test_recall = recall_score(y_test, y_test_pred_xgb)
    xgb_test_f1 = f1_score(y_test, y_test_pred_xgb)
    xgb_test_conf_matrix = confusion_matrix(y_test, y_test_pred_xgb)
    xgb_test_class_report = classification_report(y_test, y_test_pred_xgb)

    print("\nXGBoost Test Results:")
    print(f"Accuracy: {xgb_test_accuracy:.4f}")
    print(f"Precision: {xgb_test_precision:.4f}")
    print(f"Recall: {xgb_test_recall:.4f}")
    print(f"F1-Score: {xgb_test_f1:.4f}")
    print("Confusion Matrix:\n", xgb_test_conf_matrix)
    print("Classification Report:\n", xgb_test_class_report)


XGBoost Test Results:
Accuracy: 0.7237
Precision: 0.3334
Recall: 0.3725
F1-Score: 0.3519
Confusion Matrix:
 [[14340  3315]
 [ 2793  1658]]
Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.81      0.82     17655
           1       0.33      0.37      0.35      4451

    accuracy                           0.72     22106
   macro avg       0.59      0.59      0.59     22106
weighted avg       0.74      0.72      0.73     22106



In [11]:
# =============================================================================
# 9. TabPFN Modeling Analysis
# =============================================================================
X_train_tabpfn = X_train_cls.copy()
y_train_tabpfn = y_train_cls.copy()

# Downsample if the dataset is too large.
max_samples = 10_000
if X_train_tabpfn.shape[0] > max_samples:
    print(f"Downsampling from {X_train_tabpfn.shape[0]} to {max_samples} samples for TabPFN.")
    sampled_indices = np.random.choice(X_train_tabpfn.shape[0], size=max_samples, replace=False)
    X_train_tabpfn = X_train_tabpfn[sampled_indices]
    y_train_tabpfn = np.array(y_train_tabpfn)[sampled_indices]

# Train TabPFN.
tabpfn_model = TabPFNClassifier(device='cuda', random_state=42, ignore_pretraining_limits=True)
tabpfn_model.fit(X_train_tabpfn, y_train_tabpfn)

if 'No-show' in test.columns:
    X_test_tabpfn = X_test_processed
    y_test = test['No-show']

    y_test_pred_tabpfn = tabpfn_model.predict(X_test_tabpfn)
    tabpfn_test_accuracy = accuracy_score(y_test, y_test_pred_tabpfn)
    tabpfn_test_precision = precision_score(y_test, y_test_pred_tabpfn)
    tabpfn_test_recall = recall_score(y_test, y_test_pred_tabpfn)
    tabpfn_test_f1 = f1_score(y_test, y_test_pred_tabpfn)
    tabpfn_test_conf_matrix = confusion_matrix(y_test, y_test_pred_tabpfn)
    tabpfn_test_class_report = classification_report(y_test, y_test_pred_tabpfn)

    print("\nTabPFN Test Results:")
    print(f"Accuracy: {tabpfn_test_accuracy:.4f}")
    print(f"Precision: {tabpfn_test_precision:.4f}")
    print(f"Recall: {tabpfn_test_recall:.4f}")
    print(f"F1-Score: {tabpfn_test_f1:.4f}")
    print("Confusion Matrix:\n", tabpfn_test_conf_matrix)
    print("Classification Report:\n", tabpfn_test_class_report)

Downsampling from 141106 to 10000 samples for TabPFN.


/usr/local/lib/python3.11/dist-packages/tabpfn/base.py:88: UserWarning: Downloading model to /root/.cache/tabpfn/tabpfn-v2-classifier.ckpt.
  model, _, config_ = load_model_criterion_config(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tabpfn-v2-classifier.ckpt:   0%|          | 0.00/29.0M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]


TabPFN Test Results:
Accuracy: 0.6607
Precision: 0.3272
Recall: 0.6486
F1-Score: 0.4350
Confusion Matrix:
 [[11718  5937]
 [ 1564  2887]]
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.66      0.76     17655
           1       0.33      0.65      0.43      4451

    accuracy                           0.66     22106
   macro avg       0.60      0.66      0.60     22106
weighted avg       0.77      0.66      0.69     22106



In [12]:
# =============================================================================
# 10. AutoTabPFN Modeling Analysis
# =============================================================================
X_train_autotabpfn = X_train_cls.copy()
y_train_autotabfn = y_train_cls.copy()

# Downsample if necessary.
if X_train_autotabpfn.shape[0] > max_samples:
    print(f"Downsampling from {X_train_autotabpfn.shape[0]} to {max_samples} samples for AutoTabPFN.")
    sampled_indices = np.random.choice(X_train_autotabpfn.shape[0], size=max_samples, replace=False)
    X_train_autotabpfn = X_train_autotabpfn[sampled_indices]
    y_train_autotabfn = np.array(y_train_autotabfn)[sampled_indices]

# Initialize and train AutoTabPFN.
auto_tabpfn_model = AutoTabPFNClassifier(
    device='cuda',     # Use GPU if available
    max_time=120,      # Maximum search time in seconds
    random_state=42
)
print("Training AutoTabPFN...")
auto_tabpfn_model.fit(X_train_autotabpfn, y_train_autotabfn)

if 'No-show' in test.columns:
    X_test_autotabpfn = X_test_processed
    y_test = test['No-show']

    y_test_pred_auto = auto_tabpfn_model.predict(X_test_autotabpfn)
    auto_test_accuracy = accuracy_score(y_test, y_test_pred_auto)
    auto_test_precision = precision_score(y_test, y_test_pred_auto, average='weighted')
    auto_test_recall = recall_score(y_test, y_test_pred_auto, average='weighted')
    auto_test_f1 = f1_score(y_test, y_test_pred_auto, average='weighted')

    auto_conf_matrix = confusion_matrix(y_test, y_test_pred_auto)

    print("\nAutoTabPFN Test Results:")
    print(f"Accuracy: {auto_test_accuracy:.4f}")
    print(f"Precision: {auto_test_precision:.4f}")
    print(f"Recall: {auto_test_recall:.4f}")
    print(f"F1-Score: {auto_test_f1:.4f}")
    print("\nConfusion Matrix:\n", auto_conf_matrix)
    print("\nClassification Report:\n", classification_report(y_test, y_test_pred_auto))

Downsampling from 141106 to 10000 samples for AutoTabPFN.
Training AutoTabPFN...


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tabpfn/base.py:88: UserWarning: Downloading model to /usr/local/lib/python3.11/dist-packages/tabpfn_extensions/hpo/hpo_models/tabpfn-v2-classifier-od3j1g5m.ckpt.
  model, _, config_ = load_model_criterion_config(


tabpfn-v2-classifier-od3j1g5m.ckpt:   0%|          | 0.00/29.0M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/tabpfn/base.py:88: UserWarning: Downloading model to /usr/local/lib/python3.11/dist-packages/tabpfn_extensions/hpo/hpo_models/tabpfn-v2-classifier-llderlii.ckpt.
  model, _, config_ = load_model_criterion_config(


...

/usr/local/lib/python3.11/dist-packages/tabpfn/base.py:88: UserWarning: Downloading model to /usr/local/lib/python3.11/dist-packages/tabpfn_extensions/hpo/hpo_models/tabpfn-v2-classifier-znskzxi4.ckpt.
  model, _, config_ = load_model_criterion_config(


tabpfn-v2-classifier-znskzxi4.ckpt:   0%|          | 0.00/12.9M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Order of selections: [0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0]
INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Val loss over iterations: [-0.7937018806935058, -0.7946447987070232, -0.7948497649133118, -0.7948497649133118, -0.7948707023214812, -0.7948707023214812, -0.7948707023214812, -0.7948809873640905, -0.7948809873640905, -0.7948809873640905, -0.7948997208345576, -0.7948997208345576, -0.7948997208345576, -0.7949022920952101, -0.7949022920952101, -0.7949022920952101, -0.7949022920952101, -0.7949022920952101, -0.7949022920952101, -0.7949022920952101, -0.7949022920952101, -0.7949022920952101, -0.7949022920952101, -0.7949022920952101, -0.7949022920952101]
INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Model losses: [-0.79370188 -0.79014473 -0.78864935 -0.78387379]
INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Best weights: [0.64


AutoTabPFN Test Results:
Accuracy: 0.6334
Precision: 0.7790
Recall: 0.6334
F1-Score: 0.6696

Confusion Matrix:
 [[10816  6839]
 [ 1265  3186]]

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.61      0.73     17655
           1       0.32      0.72      0.44      4451

    accuracy                           0.63     22106
   macro avg       0.61      0.66      0.58     22106
weighted avg       0.78      0.63      0.67     22106

